# Verity: cloud run, part 2 (Qwen2.5-32B)

Run on the same A100 server after part 1 has finished (about 30-45 minutes). Data and part-1 results on the server are kept; the last cell prints all experiment-8 results.

In [10]:
!nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv

name, memory.total [MiB], memory.used [MiB]
NVIDIA A100-SXM4-80GB, 81920 MiB, 0 MiB


In [11]:
# unpack the project code carried inside this notebook
import base64, io, zipfile, os
CODE = '...'
os.makedirs('/content/verity', exist_ok=True)
zipfile.ZipFile(io.BytesIO(base64.b64decode(CODE))).extractall('/content/verity')
os.chdir('/content/verity')
print(sorted(os.listdir('.')))

['data', 'experiments', 'figures', 'models', 'results', 'verity']


In [12]:
!pip -q install 'transformers==4.46.3' 'sentence-transformers==3.3.1' 'tokenizers<0.21' 'accelerate==1.1.1' 'datasets==3.1.0' 'huggingface-hub<1.0' bm25s==0.3.15 PyStemmer scikit-learn scipy pandas python-dotenv
%cd /content/verity
!python -m verity.data

/content
SciFact corpus 5183 | train 809 | dev 300
Climate-FEVER claims 1381 | evidence pool 5240
RAGTruth train 15090 | test 2700


In [13]:
# Check: is the 14B judge's low score a bug? (raw answers + 50 claims unbatched, ~5 min)
!python -m experiments.e08_scale_up --only=check14 2>&1 | grep -v -i warn | tail -20

I0000 00:00:1793129517.823936    7264 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1793129517.896956    7264 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1793129520.429519    7264 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1793129520.430055    7264 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.

In [14]:
# C1: Qwen2.5-32B as a claim judge (SciFact dev + Climate-FEVER sample)
!python -m experiments.e08_scale_up --only=judge32 2>&1 | grep -v -i warn | tail -20

  judge Qwen2.5-32B-Instruct: 184/300 generated
  judge Qwen2.5-32B-Instruct: 192/300 generated
  judge Qwen2.5-32B-Instruct: 200/300 generated
  judge Qwen2.5-32B-Instruct: 208/300 generated
  judge Qwen2.5-32B-Instruct: 216/300 generated
  judge Qwen2.5-32B-Instruct: 224/300 generated
  judge Qwen2.5-32B-Instruct: 232/300 generated
  judge Qwen2.5-32B-Instruct: 240/300 generated
  judge Qwen2.5-32B-Instruct: 248/300 generated
  judge Qwen2.5-32B-Instruct: 256/300 generated
  judge Qwen2.5-32B-Instruct: 264/300 generated
  judge Qwen2.5-32B-Instruct: 272/300 generated
  judge Qwen2.5-32B-Instruct: 280/300 generated
  judge Qwen2.5-32B-Instruct: 288/300 generated
  judge Qwen2.5-32B-Instruct: 296/300 generated
  judge Qwen2.5-32B-Instruct: 300/300 generated
qwen2.5-32b climate_fever_sample 0.545 {'SUPPORTED': 18, 'REFUTED': 86, 'NEI': 196}
RESULTS-JSON-START
{"gpu": "NVIDIA A100-SXM4-80GB", "large_verifier": "MoritzLaurer/DeBERTa-v3-large-mnli-fever-anli-ling-wanli", "large_runs": [{"s

In [15]:
# C2: Qwen2.5-32B judging whole RAGTruth answers against their sources
!python -m experiments.e08_scale_up --only=ragtruth32 2>&1 | grep -v -i warn | tail -20

  answer judge local:Qwen/Qwen2.5-32B-Instruct: 180/240 generated
  answer judge local:Qwen/Qwen2.5-32B-Instruct: 184/240 generated
  answer judge local:Qwen/Qwen2.5-32B-Instruct: 188/240 generated
  answer judge local:Qwen/Qwen2.5-32B-Instruct: 192/240 generated
  answer judge local:Qwen/Qwen2.5-32B-Instruct: 196/240 generated
  answer judge local:Qwen/Qwen2.5-32B-Instruct: 200/240 generated
  answer judge local:Qwen/Qwen2.5-32B-Instruct: 204/240 generated
  answer judge local:Qwen/Qwen2.5-32B-Instruct: 208/240 generated
  answer judge local:Qwen/Qwen2.5-32B-Instruct: 212/240 generated
  answer judge local:Qwen/Qwen2.5-32B-Instruct: 216/240 generated
  answer judge local:Qwen/Qwen2.5-32B-Instruct: 220/240 generated
  answer judge local:Qwen/Qwen2.5-32B-Instruct: 224/240 generated
  answer judge local:Qwen/Qwen2.5-32B-Instruct: 228/240 generated
  answer judge local:Qwen/Qwen2.5-32B-Instruct: 232/240 generated
  answer judge local:Qwen/Qwen2.5-32B-Instruct: 236/240 generated
  answer j

In [16]:
# the results, printed so they are saved in this notebook
import json
r = json.load(open('/content/verity/results/e08_scale_up.json'))
print('RESULTS-JSON-START')
print(json.dumps(r))
print('RESULTS-JSON-END')

RESULTS-JSON-START
{"gpu": "NVIDIA A100-SXM4-80GB", "large_verifier": "MoritzLaurer/DeBERTa-v3-large-mnli-fever-anli-ling-wanli", "large_runs": [{"seed": 42, "calibration": {"n": 161, "accuracy": 0.8633540372670807, "macro_f1": 0.8604024459891972, "per_class": {"SUPPORTED": {"precision": 0.9206349206349206, "recall": 0.8405797101449275, "f1": 0.8787878787878788, "support": 69}, "REFUTED": {"precision": 0.9615384615384616, "recall": 0.7575757575757576, "f1": 0.847457627118644, "support": 33}, "NEI": {"precision": 0.7777777777777778, "recall": 0.9491525423728814, "f1": 0.8549618320610687, "support": 59}}, "macro_f1_ci95": [0.8001260475126473, 0.9126711692661376], "ece": 0.09713566784532915}, "scifact_dev": {"n": 300, "accuracy": 0.8333333333333334, "macro_f1": 0.8267327717506542, "per_class": {"SUPPORTED": {"precision": 0.9433962264150944, "recall": 0.8064516129032258, "f1": 0.8695652173913043, "support": 124}, "REFUTED": {"precision": 0.8545454545454545, "recall": 0.734375, "f1": 0.7899